In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,StratifiedKFold,GridSearchCV
from sklearn.metrics import classification_report
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


warnings.filterwarnings('ignore')


In [4]:
# importing data
from ucimlrepo import fetch_ucirepo
heart_disease = fetch_ucirepo(id=45)
X = heart_disease.data.features
y = heart_disease.data.targets

In [5]:
## handling missing values
print(X.isna().sum())

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
dtype: int64


In [6]:
X['ca'].fillna(X['ca'].mode()[0],inplace = True) # filling by mode as ca and thal are categorical values.
X['thal'].fillna(X['thal'].mode()[0],inplace = True)

In [7]:
print(X.isna().sum())

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
dtype: int64


In [8]:
# converting to binary classification
def function(row):
    if row >=1:
        return 1
    else:
        return 0

y['num'] = y['num'].apply(function)

In [9]:
print(y.value_counts())


num
0      164
1      139
Name: count, dtype: int64


In [10]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,stratify = y)
y_test = y_test.reset_index(drop=True)
X_test_original = X_test.copy()

In [11]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)
print(y_train)

     num
231    1
32     1
181    1
125    0
269    0
..   ...
80     0
84     0
240    0
270    1
298    1

[242 rows x 1 columns]


In [24]:
skf=StratifiedKFold(n_splits=10,shuffle=False)

param_grid1={'n_estimators' : [50,100,150,200],'max_depth' : [5,10,15,20]}
param_grid_lr = {'C': [0.01, 0.1, 1, 10, 100]}

model = LogisticRegression()
model1 = RandomForestClassifier()
model2 = XGBClassifier()

gridsearch_lr = GridSearchCV(estimator=model, param_grid=param_grid_lr, cv=skf,scoring='recall', n_jobs=-1)
gridsearch1= GridSearchCV(estimator = model1,param_grid = param_grid1,scoring='recall',n_jobs=-1,cv=skf)
gridsearch2 = GridSearchCV(estimator = model2,param_grid = param_grid1,scoring='recall',n_jobs=-1,cv=skf)

# fitting the model
gridsearch_lr.fit(X_train,y_train.values)
gridsearch1.fit(X_train,y_train.values)
gridsearch2.fit(X_train,y_train.values)

print(gridsearch_lr.best_params_)
print(gridsearch1.best_params_)
print(gridsearch2.best_params_)

print(gridsearch_lr.best_score_)
print(gridsearch1.best_score_)
print(gridsearch2.best_score_)






{'C': 0.1}
{'max_depth': 5, 'n_estimators': 50}
{'max_depth': 10, 'n_estimators': 100}
0.7643939393939394
0.7651515151515152
0.7651515151515151


In [22]:
### Predictions and testing
predictions = gridsearch_lr.predict(X_test)
predictions1 = gridsearch1.predict(X_test)
predictions2 = gridsearch2.predict(X_test)

print("Final Results of the best models")
print("----------------------------------\n")
print("Logistic Regression")
print("----------------------------------")
print(classification_report(y_test,predictions))
print()
print("Random Forest Classifier")
print("----------------------------------")
print(classification_report(y_test,predictions1))
print()
print("XGBoost Classifier")
print("----------------------------------")
print(classification_report(y_test,predictions2))




Final Results of the best models
----------------------------------

Logistic Regression
----------------------------------
              precision    recall  f1-score   support

           0       0.89      0.97      0.93        33
           1       0.96      0.86      0.91        28

    accuracy                           0.92        61
   macro avg       0.92      0.91      0.92        61
weighted avg       0.92      0.92      0.92        61


Random Forest Classifier
----------------------------------
              precision    recall  f1-score   support

           0       0.79      0.91      0.85        33
           1       0.87      0.71      0.78        28

    accuracy                           0.82        61
   macro avg       0.83      0.81      0.81        61
weighted avg       0.83      0.82      0.82        61


XGBoost Classifier
----------------------------------
              precision    recall  f1-score   support

           0       0.78      0.94      0.85        

here as you can see the best recall scores across all three models in cv set is only 0.76 so that just means that we finding an optimistic test set and the actual recall scores are 0.76 and here data is the primary issue.

In [ ]:

y_test = np.array(y_test.values)


In [34]:
# Error analysis 
y_test = y_test.flatten()
wrong = pd.DataFrame(X_test_original[predictions!=y_test])


In [35]:
wrong['actual'] = y_test[predictions != y_test]
wrong['predicted'] = predictions[predictions != y_test]


In [36]:
print(wrong)

     age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  \
211   38    1   1       120   231    0        0      182      1      3.8   
250   57    1   4       110   201    0        0      126      1      1.5   
92    62    1   3       130   231    0        0      146      0      1.8   
178   43    1   3       130   315    0        0      162      0      1.9   
294   63    0   4       124   197    0        0      136      1      0.0   
298   45    1   1       110   264    0        0      132      0      1.2   
266   52    1   4       128   204    1        0      156      1      1.0   
141   59    1   1       170   288    0        2      159      0      0.2   
199   59    1   1       160   273    0        2      125      0      0.0   

     slope   ca  thal  actual  predicted  
211      2  0.0   7.0       1          0  
250      2  0.0   6.0       0          1  
92       2  3.0   7.0       0          1  
178      1  1.0   3.0       0          1  
294      2  0.0   3.0   

In [37]:
### Lets compare it to the predicitons that were right 
right = pd.DataFrame(X_test_original[predictions==y_test][:15])
right['actual'] = y_test[predictions==y_test][:15]
right['predicted']=predictions[predictions==y_test][:15]


In [38]:
print(right)

     age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  \
29    40    1   4       110   167    0        2      114      1      2.0   
197   45    0   4       138   236    0        2      152      1      0.2   
112   52    1   1       118   186    0        2      190      0      0.0   
290   67    1   3       152   212    0        2      150      0      0.8   
102   57    0   4       128   303    0        2      159      0      0.0   
220   41    0   3       112   268    0        2      172      1      0.0   
230   52    0   3       136   196    0        2      169      0      0.1   
192   43    1   4       132   247    1        2      143      1      0.1   
17    54    1   4       140   239    0        0      160      0      1.2   
45    58    1   3       112   230    0        2      165      0      2.5   
163   58    0   4       100   248    0        2      122      0      1.0   
279   58    0   4       130   197    0        0      131      0      0.6   
244   60    

The model struggles most with patients whose thal and ca values contradict their actual diagnosis. These are edge cases where even a doctor would want more information.